# FreeFlyer Same File Check

This notebook compares two FreeFlyer result folders using `InContact.txt` and `IsNight.txt`.

The first cell calculates total contact time and invalid nights for each telescope. A night is valid if it contains at least 30 continuous minutes of contact, with a possible numerical tolerance. Every missed night after the first consecutive missed night is invalid.

The second cell checks the raw files, time grids and Boolean states, then identifies the specific nights responsible for any differences.

Version 07.2026 by Pedro de S. C. Leonardo

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# Settings to change
FOLDER_1 = Path("All_Results/Scheduling/Results_a28750_i40")
FOLDER_2 = Path("All_Results/Testing/Results_a28750_i40")

DAYS = 365
TELESCOPES = ["Chile", "La Palma", "Hawaii", "SALT", "DAG", "HARPS", "CORALIE"]

MIN_VALID_CONTACT_H = 0.5
MAX_CONSECUTIVE_MISSED_NIGHTS = 1
EXCLUDE_PARTIAL_EDGE_NIGHTS = True
CONTACT_DURATION_TOLERANCE_S = 0.0


def read_result(folder, filename):
    path = folder / filename

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    return pd.read_csv(path, skiprows=3, sep=r"\s+", header=None).to_numpy(dtype=float)


def infer_simulation_end(time):
    if len(time) < 2:
        raise ValueError("At least two time samples are required.")

    time_steps = np.diff(time)

    if not np.all(time_steps > 0):
        raise ValueError("Time samples must be strictly increasing.")

    return time[-1] + np.median(time_steps)


def get_boolean_intervals(time, state, simulation_end):
    state = np.asarray(state, dtype=bool)
    changes = np.diff(np.r_[False, state, False].astype(int))
    start_indices = np.where(changes == 1)[0]
    end_indices = np.where(changes == -1)[0]
    starts = time[start_indices]
    ends = np.array([time[index] if index < len(time) else simulation_end for index in end_indices])

    return starts, ends


def get_night_intervals(time, is_night, simulation_end):
    starts, ends = get_boolean_intervals(time, is_night, simulation_end)
    keep = np.ones(len(starts), dtype=bool)

    if EXCLUDE_PARTIAL_EDGE_NIGHTS and len(starts) > 0:
        if is_night[0]:
            keep[0] = False
        if is_night[-1]:
            keep[-1] = False

    return starts[keep], ends[keep]


def get_missed_night_runs(night_has_valid_contact):
    runs = []
    run_start = None

    for night_index, has_contact in enumerate(night_has_valid_contact):
        if not has_contact and run_start is None:
            run_start = night_index
        elif has_contact and run_start is not None:
            runs.append((run_start, night_index - 1))
            run_start = None

    if run_start is not None:
        runs.append((run_start, len(night_has_valid_contact) - 1))

    return runs


def calculate_telescope_statistics(time, visible, is_night, simulation_end):
    contact_starts, contact_ends = get_boolean_intervals(time, visible, simulation_end)
    total_contact_h = np.sum(contact_ends - contact_starts) * 24
    night_starts, night_ends = get_night_intervals(time, is_night, simulation_end)

    if len(night_starts) == 0:
        raise ValueError("No complete nighttime intervals were found.")

    night_has_valid_contact = np.zeros(len(night_starts), dtype=bool)

    for night_index, (night_start, night_end) in enumerate(zip(night_starts, night_ends)):
        overlap_starts = np.maximum(contact_starts, night_start)
        overlap_ends = np.minimum(contact_ends, night_end)
        overlap_durations_s = np.maximum(0, overlap_ends - overlap_starts) * 86400

        night_has_valid_contact[night_index] = np.any(
            overlap_durations_s >= MIN_VALID_CONTACT_H * 3600 - CONTACT_DURATION_TOLERANCE_S)

    missed_runs = get_missed_night_runs(night_has_valid_contact)
    invalid_nights = sum(
        max(0, end - start + 1 - MAX_CONSECUTIVE_MISSED_NIGHTS) for start, end in missed_runs)

    return {
        "Total contact [h]": total_contact_h,
        "Invalid nights": invalid_nights}


def analyse_folder(folder):
    contact_data = read_result(folder, "InContact.txt")
    night_data = read_result(folder, "IsNight.txt")

    contact_data = contact_data[(contact_data[:, 0] >= 0) & (contact_data[:, 0] < DAYS)]
    night_data = night_data[(night_data[:, 0] >= 0) & (night_data[:, 0] < DAYS)]

    if contact_data.shape != night_data.shape:
        raise ValueError(f"InContact.txt and IsNight.txt have different shapes in {folder}.")

    time = contact_data[:, 0]

    if not np.allclose(time, night_data[:, 0], rtol=0, atol=1e-10):
        raise ValueError(f"The time samples do not match in {folder}.")

    if contact_data.shape[1] != len(TELESCOPES) + 1:
        raise ValueError(
            f"{folder} contains {contact_data.shape[1] - 1} telescope columns, "
            f"but {len(TELESCOPES)} telescope names were provided.")

    simulation_end = min(DAYS, infer_simulation_end(time))
    rows = []

    for column, telescope in enumerate(TELESCOPES, start=1):
        visible = contact_data[:, column] >= 0.5
        is_night = night_data[:, column] >= 0.5
        statistics = calculate_telescope_statistics(time, visible, is_night, simulation_end)
        rows.append({"Telescope": telescope, **statistics})

    return pd.DataFrame(rows)


folder_1_results = analyse_folder(FOLDER_1)
folder_2_results = analyse_folder(FOLDER_2)

comparison = folder_1_results.merge(
    folder_2_results, on="Telescope", suffixes=(" – Folder 1", " – Folder 2"))

comparison["Contact difference [h]"] = comparison["Total contact [h] – Folder 2"] - comparison["Total contact [h] – Folder 1"]
comparison["Invalid-night difference"] = comparison["Invalid nights – Folder 2"] - comparison["Invalid nights – Folder 1"]

comparison["Identical"] = (
    np.isclose(
        comparison["Total contact [h] – Folder 1"],
        comparison["Total contact [h] – Folder 2"],
        rtol=0,
        atol=CONTACT_DURATION_TOLERANCE_S / 3600) &
    (comparison["Invalid nights – Folder 1"] == comparison["Invalid nights – Folder 2"]))

display(comparison.round(6).style.hide(axis="index"))

if comparison["Identical"].all():
    print("All results are identical.")
else:
    print("The results are not identical.")

Telescope,Total contact [h] – Folder 1,Invalid nights – Folder 1,Total contact [h] – Folder 2,Invalid nights – Folder 2,Contact difference [h],Invalid-night difference,Identical
Chile,354.750000,143,354.750000,143,0.000000,0,True
La Palma,293.716667,154,293.716667,154,0.000000,0,True
Hawaii,426.883333,129,426.883333,129,0.000000,0,True
SALT,212.233333,174,212.233333,174,0.000000,0,True
DAG,195.458333,170,195.458333,170,0.000000,0,True
HARPS,270.908333,159,270.908333,159,0.000000,0,True
CORALIE,270.950000,159,270.950000,159,0.000000,0,True


All results are identical.


In [2]:
import filecmp

# Check raw files and Boolean samples
difference_rows = []

for filename in ["InContact.txt", "IsNight.txt"]:
    data_1 = read_result(FOLDER_1, filename)
    data_2 = read_result(FOLDER_2, filename)

    data_1 = data_1[(data_1[:, 0] >= 0) & (data_1[:, 0] < DAYS)]
    data_2 = data_2[(data_2[:, 0] >= 0) & (data_2[:, 0] < DAYS)]

    print(f"\n{filename}")
    print("Byte-for-byte identical:", filecmp.cmp(
        FOLDER_1 / filename, FOLDER_2 / filename, shallow=False))

    if data_1.shape != data_2.shape:
        print(f"Different shapes: {data_1.shape} and {data_2.shape}")
        continue

    time_difference_s = np.max(np.abs(data_2[:, 0] - data_1[:, 0])) * 86400
    print(f"Maximum time-grid difference: {time_difference_s:.12g} s")

    for column, telescope in enumerate(TELESCOPES, start=1):
        state_1 = data_1[:, column] >= 0.5
        state_2 = data_2[:, column] >= 0.5
        different_indices = np.flatnonzero(state_1 != state_2)

        difference_rows.append({
            "File": filename,
            "Telescope": telescope,
            "Different samples": len(different_indices),
            "First difference [day]": data_1[different_indices[0], 0] if len(different_indices) else np.nan})

sample_differences = pd.DataFrame(difference_rows)

print("\nBoolean-state differences")
display(sample_differences.style.hide(axis="index"))


def get_night_diagnostics(folder, telescope):
    contact_data = read_result(folder, "InContact.txt")
    night_data = read_result(folder, "IsNight.txt")

    contact_data = contact_data[(contact_data[:, 0] >= 0) & (contact_data[:, 0] < DAYS)]
    night_data = night_data[(night_data[:, 0] >= 0) & (night_data[:, 0] < DAYS)]

    time = contact_data[:, 0]
    simulation_end = min(DAYS, infer_simulation_end(time))
    column = TELESCOPES.index(telescope) + 1

    visible = contact_data[:, column] >= 0.5
    is_night = night_data[:, column] >= 0.5

    contact_starts, contact_ends = get_boolean_intervals(time, visible, simulation_end)
    night_starts, night_ends = get_night_intervals(time, is_night, simulation_end)

    rows = []
    missed_streak = 0

    for night_number, (night_start, night_end) in enumerate(zip(night_starts, night_ends), start=1):
        overlap_starts = np.maximum(contact_starts, night_start)
        overlap_ends = np.minimum(contact_ends, night_end)
        durations_s = np.maximum(0, overlap_ends - overlap_starts) * 86400
        longest_contact_s = durations_s.max() if len(durations_s) else 0
        valid = longest_contact_s >= MIN_VALID_CONTACT_H * 3600 - CONTACT_DURATION_TOLERANCE_S

        if valid:
            missed_streak = 0
        else:
            missed_streak += 1

        invalid = missed_streak > MAX_CONSECUTIVE_MISSED_NIGHTS

        rows.append({
            "Night": night_number,
            "Start [day]": night_start,
            "End [day]": night_end,
            "Longest contact [min]": longest_contact_s / 60,
            "Valid": valid,
            "Missed streak": missed_streak,
            "Invalid": invalid})

    return pd.DataFrame(rows)


# Show only nights whose classification differs
for telescope in TELESCOPES:
    nights_1 = get_night_diagnostics(FOLDER_1, telescope)
    nights_2 = get_night_diagnostics(FOLDER_2, telescope)

    night_comparison = nights_1.merge(
        nights_2,
        on="Night",
        how="outer",
        suffixes=(" – Folder 1", " – Folder 2"),
        indicator=True)

    different = night_comparison[
        (night_comparison["_merge"] != "both") |
        (night_comparison["Valid – Folder 1"] != night_comparison["Valid – Folder 2"]) |
        (night_comparison["Invalid – Folder 1"] != night_comparison["Invalid – Folder 2"])].drop(columns="_merge")

    if not different.empty:
        print(f"\nClassification differences: {telescope}")
        display(different.round(9).style.hide(axis="index"))


InContact.txt
Byte-for-byte identical: False
Maximum time-grid difference: 0 s

IsNight.txt
Byte-for-byte identical: False
Maximum time-grid difference: 0 s

Boolean-state differences


File,Telescope,Different samples,First difference [day]
InContact.txt,Chile,0,nan
InContact.txt,La Palma,0,nan
InContact.txt,Hawaii,0,nan
InContact.txt,SALT,0,nan
InContact.txt,DAG,0,nan
InContact.txt,HARPS,0,nan
InContact.txt,CORALIE,0,nan
IsNight.txt,Chile,0,nan
IsNight.txt,La Palma,0,nan
IsNight.txt,Hawaii,0,nan
